[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RadimKozl/JLNN/blob/main/examples/JLNN_vit_training.ipynb)

# **JLNN + Vision Transformer: Neuro-symbolic image classification with explanation and uncertainty**

## **Overview**
This notebook demonstrates the integration of a **Vision Transformer (ViT)** backbone with a **Justifiable Logical Neural Network (JLNN)** layer. Unlike standard black-box models, this hybrid architecture provides both high-performance visual feature extraction and interpretable logical reasoning.

### **Core Components**
1. **ViT Backbone:** Extracts high-level semantic features from images using self-attention mechanisms.
2. **Fuzzy Grounding:** Maps the continuous feature space (CLS token) into logical predicates (fuzzy truth values).
3. **JLNN Layer:** Implements Łukasiewicz t-norm logic to process these predicates through a set of human-defined rules.
4. **Uncertainty Quantification:** Uses **[L, U] intervals** (Lower and Upper bounds) to represent not just the probability, but the *certainty* of the model's decision.

### **Key Features**
- **Explainability:** Each classification is backed by a "Logical Audit" showing which rules were triggered.
- **Robustness:** Logical constraints help regularize the network and prevent overconfident incorrect predictions.
- **Hardware Monitoring:** Real-time throughput and resource utilization tracking via TensorBoard.
- **Persistent Storage:** Automated checkpointing to Hugging Face Hub for experiment reproducibility.

### **Advanced Hardware Profiling (XProf)**
To optimize JAX performance on accelerators (TPU/GPU), we utilize the `jax.profiler`. This captures low-level execution details:
- **Step Time Breakdown:** How much time is spent on Python overhead vs. actual device execution.
- **HLO Op Profile:** Identification of expensive XLA operations.
- **Memory Bandwidth:** Monitoring if the model is bottlenecked by data transfer.

**How to view:**
1. Run the training cell.
2. In the TensorBoard interface, click the dropdown menu (usually "Inactive"

------------------------------------
## ***Instalation & Imports***
------------------------------------

In [ ]:
def setup_jlnn_tpu_environment():
    print("🧹 Cleaning environment from potential conflicts...")
    # We will remove the old JAX to force a clean TPU installation
    !pip uninstall -y jax jaxlib torch --quiet

    print("🚀 Installing JAX with TPU support...")
    # Install TPU specific version
    !pip install "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html --quiet

    print("📦 Installing jax-lnn, visual tools and others tools...")
    # Installing your project directly from GitHub
    !pip install git+https://github.com/RadimKozl/JLNN.git --quiet
    !pip install seaborn xarray --quiet
    !pip install grain==0.2.16 orbax-checkpoint optax --quiet
    !pip install orbax-export --quiet
    !pip install pillow --quiet
    !pip install kagglehub --quiet
    !pip install tqdm --quiet
    !pip install pandas --quiet
    !pip install opencv-contrib-python-headless --quiet
    !pip install huggingface_hub --quiet
    !pip install torch tensorboard --quiet
    !pip install jax2onnx onnxruntime --quiet
    !pip install tensorflow tf2onnx --quiet
    !pip install xprof --quite
    !pip install -U tensorboard-plugin-profile --quiet

    print("\n🔄 RESTARTING KERNEL to apply TPU changes...")
    os.kill(os.getpid(), 9)

In [ ]:
try:
    import jlnn
    import os
    import jax
    # Checking if we see TPU
    if 'TPU_NAME' in os.environ:
        import jax.tools.colab_tpu
        jax.tools.colab_tpu.setup_tpu()

    print(f"✅ jax-lnn is ready.")
    print(f"✅ Devices: {jax.devices()}")
    # Here you should see: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), ...]

except (ImportError, RuntimeError):
    import os
    setup_jlnn_tpu_environment()

In [ ]:
!rm -rf /content/sample_data

In [ ]:
!apt install git-lfs

In [ ]:
import jax
import os

if 'TPU_NAME' in os.environ:
    import jax.tools.colab_tpu
    jax.tools.colab_tpu.setup_tpu()

print(f"Confirmed Devices: {jax.devices()}")

In [ ]:
import os, glob
import subprocess
import datetime
import shutil

import jax
from jax import export
import time
import pickle
import json
import jax.numpy as jnp
from flax import nnx
from jax.export import export
import onnx
from onnx import helper, TensorProto, numpy_helper
from typing import Any

import optax


from jax2onnx import to_onnx
import onnxruntime as ort
import jax.profiler
from jax.experimental import jax2tf

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import seaborn as sns
import tf2onnx
import tensorflow as tf

import orbax.checkpoint as ocp
import orbax.export
import grain
import grain.python as grainp
print('grain version: ', grain.__version__)
from grain import transforms as gt

from PIL import Image

# JLNN imports
from jlnn.symbolic.compiler import LNNFormula
from jlnn.nn.constraints import apply_constraints, clip_predicates
from jlnn.training.losses import total_lnn_loss
from jlnn.export.stablehlo import export_to_stablehlo, save_stablehlo_artifact
from jlnn.export.data import extract_logic_parameters

from huggingface_hub import (
    get_full_repo_name,
    login,
    upload_folder,
    hf_hub_download,
    snapshot_download,
    HfApi
)

from torch.utils.tensorboard import SummaryWriter

from google.colab import userdata
from google.colab import files

------------------------------------
## ***Git settings***
------------------------------------

In [ ]:
def set_git_config(email, name):
    try:
        # Setting global user.email
        subprocess.run(["git", "config", "--global", "user.email", email], check=True)
        print(f"Git user.email set to: {email}")

        # Setting the global user.name
        subprocess.run(["git", "config", "--global", "user.name", name], check=True)
        print(f"Git user.name set to: {name}")

        # Check settings (optional)
        email_output = subprocess.run(["git", "config", "--global", "user.email"], capture_output=True, text=True, check=True)
        name_output = subprocess.run(["git", "config", "--global", "user.name"], capture_output=True, text=True, check=True)
        print(f"Check - Email: {email_output.stdout.strip()}")
        print(f"Check - Name: {name_output.stdout.strip()}")

    except subprocess.CalledProcessError as e:
        print(f"Error while setting up Git configuration: {e}")

In [ ]:
user_email = userdata.get('USER_EMAIL')
user_name = userdata.get('USER_NAME')
#print(user_email, user_name)
#set_git_config(user_email, user_name)

------------------------------------
## ***Hugging Face Settings***
------------------------------------

In [ ]:
hf_token = userdata.get('HF_TOKEN') # Hugging Face token

### === ***Hugging Face Login*** ===

In [ ]:
if hf_token:
    login(token=hf_token, add_to_git_credential=True)
    print("✅ Hugging Face login successful.")
else:
    print("❌ Error: Token 'HF_TOKEN' not found in Colab Secrets!")

In [ ]:
repo_id = "KRadim/vit-jlnn-cifar10"

In [ ]:
def save_to_hf(epoch):
    api = HfApi()
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

    print(f"📤 Sending checkpoints of epoch {epoch} to HF Hub...")

    epoch_path = os.path.join(os.path.abspath("checkpoints"), str(epoch))
    if not os.path.exists(epoch_path):
        print(f"⚠️ Checkpoint epoch {epoch} not found at {epoch_path}, skipping.")
        return

    for attempt in range(1, 4):
        try:
            api.upload_folder(
                folder_path=epoch_path,
                path_in_repo=f"checkpoints/{epoch}",
                repo_id=repo_id,
                repo_type="model",
                commit_message=f"Checkpoint epoch {epoch}",
                token=hf_token,
                ignore_patterns=["*.orbax-checkpoint-tmp*", "*__lock*"],
                delete_patterns=[],
            )
            print(f"✅ Done: https://huggingface.co/{repo_id}")
            break
        except Exception as e:
            print(f"⚠️ Attempt {attempt}/3 failed: {e}")
            time.sleep(5)

    try:
        api.upload_file(
            path_or_fileobj=f"logic_epoch_{epoch}.nc",
            path_in_repo=f"logs/logic_epoch_{epoch}.nc",
            repo_id=repo_id,
            token=hf_token
        )
    except Exception:
        print(f"ℹ️ Log logic_epoch_{epoch}.nc was not uploaded (may not exist).")


In [ ]:
def load_from_hf():
    # Downloads checkpoints from HF to a local folder
    snapshot_download(repo_id=repo_id, local_dir="./checkpoints")
    print("✅ Checkpoints downloaded from HF")

---------------------------------------------------
## ***Download the CIFAR-10 data file***
---------------------------------------------------

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blourdhuraju/cifar-10")

print("Path to dataset files:", path)

In [ ]:
!ls /root/.cache/kagglehub/datasets/blourdhuraju/cifar-10/versions/1/cifar10

### === ***Paths by structure*** ===

In [ ]:
train_dir = os.path.join(path, "cifar10", "train")
val_dir   = os.path.join(path, "cifar10", "val")
test_dir  = os.path.join(path, "cifar10", "test")

#### Load class names from class_names.txt or from train folders

In [ ]:
class_names_path = os.path.join(path, "cifar10", "class_names.txt")

In [ ]:
if os.path.exists(class_names_path):
    with open(class_names_path, "r") as f:
        class_names = [line.strip() for line in f.readlines() if line.strip()]
else:
    # fallback - from train folders
    class_names = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])

In [ ]:
num_classes = len(class_names)
class_to_idx = {name: i for i, name in enumerate(class_names)}

print(f"✅ Found {num_classes} classes: {class_names}")

### === ***Load labels from CSV (train_labels.csv, val_labels.csv, test_labels.csv)*** ===

In [ ]:
def load_labels_csv(csv_path):
    df = pd.read_csv(csv_path)
    # Assume columns: filename, label (or id, label)
    return dict(zip(df['filename'], df['label']))

#### If you have CSV, use it; otherwise fallback to folders

In [ ]:
use_csv = True

In [ ]:
if use_csv:
    train_labels = load_labels_csv(os.path.join(path, "cifar10", "train_labels.csv"))
    val_labels   = load_labels_csv(os.path.join(path, "cifar10", "val_labels.csv"))
    test_labels  = load_labels_csv(os.path.join(path, "cifar10", "test_labels.csv"))
else:
    train_labels = val_labels = test_labels = None

### === ***Creating sources*** ===

In [ ]:
def create_source(data_dir, labels_dict=None, split="train"):
    files = []
    # Iterate through class subdirectories to find images
    for class_name in class_names:
        # Correctly form the class directory path using the numerical index
        class_idx = class_to_idx[class_name]
        class_dir = os.path.join(data_dir, str(class_idx))

        if not os.path.isdir(class_dir):
            print(f"Warning: Class directory {class_dir} not found for split {split}. Skipping.")
            continue # Skip if directory does not exist

        image_files = glob.glob(os.path.join(class_dir, "*.png"))

        for f in image_files:
            # We only append the path (string) to the source to avoid shared_memory serialization issues.
            # Labels will be added later in the pipeline via a map transform.
            files.append(f)

    if not files:
        raise ValueError(f"Paths sequence can not be of 0 length. No files were added for split {split} from {data_dir}.")

    return grainp.InMemoryDataSource(files)

In [ ]:
train_source = create_source(train_dir, train_labels if use_csv else None, "train")
val_source   = create_source(val_dir,   val_labels if use_csv else None, "val")
test_source  = create_source(test_dir,  test_labels if use_csv else None, "test")

### === ***Datasets*** ===

In [ ]:
batch_size = 64

#### Define your transformation as a class. This is the "cleanest" way in Grain v2.x.

In [ ]:
class PrepareDataTransform(grain.python.MapTransform):
    def __init__(self, labels_dict):
        self.labels_dict = labels_dict

    def map(self, path):
        # This method is called for each path in the dataset
        filename = os.path.basename(path)
        label = self.labels_dict.get(filename, 0)

        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)

        # Fixed version with ImageNet normalization:
        MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

        img = img.astype(np.float32) / 255.0
        img = (img - MEAN) / STD


        return {'image': img, 'label': label}

In [ ]:
def create_dataloader(source, labels_dict, is_train=False, batch_size=64):

    # 2. Sampler
    sampler = grain.python.IndexSampler(
        num_records=len(source),
        shuffle=is_train,
        seed=42 if is_train else None,
        num_epochs=None if is_train else 1
    )

    # 3. DataLoader
    # We will use an instance of our PrepareDataTransform class instead of gt.Map
    return grain.DataLoader(
        data_source=source,
        sampler=sampler,
        operations=[
            PrepareDataTransform(labels_dict=labels_dict),
            grain.python.Batch(batch_size=batch_size, drop_remainder=is_train)
        ]
    )

### === ***Initialization*** ===

In [ ]:
train_ds = create_dataloader(train_source, train_labels, is_train=True, batch_size=64)
val_ds   = create_dataloader(val_source, val_labels,   is_train=False, batch_size=64)
test_ds  = create_dataloader(test_source, test_labels,  is_train=False, batch_size=64)

### === ***Small sample display*** ===

#### Getting a single batch using an iterator

In [ ]:
batch = next(iter(train_ds))
images = batch['image']
labels = batch['label']

#### Listing parameters

In [ ]:
print(f"Image batch shape: {images.shape}")
print(f"Labels batch shape: {labels.shape}")

#### Rendering sample images

In [ ]:
def denormalize(img):
    # ImageNet stats that you used in PrepareDataTransform
    MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    # We reverse the operation: (img * STD) + MEAN
    img = (img * STD) + MEAN
    # We trim the values ​​to stay in [0, 1] (due to floating point)
    return np.clip(img, 0, 1)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for i in range(8):
    ax = axes[i]
    label_idx = int(labels[i]) # Convert to int so it works as an index

    # Get the name (with checking if the index exists)
    name = class_names[label_idx] if label_idx < len(class_names) else f"Unknown ({label_idx})"

    display_img = denormalize(images[i])

    ax.imshow(display_img)
    ax.set_title(f"{name}") # Here we display the name
    ax.axis('off')

plt.tight_layout()
plt.show()

--------------------------------------------
## ***Model definition***
-------------------------------------------

## **Theoretical Background: JLNN and Uncertainty**

### **Interval-Valued Fuzzy Logic**
In JLNN, truth values are represented as intervals $[L, U] \subseteq [0, 1]$.
- **$L$ (Lower Bound):** Represents the verified evidence for a statement.
- **$U$ (Upper Bound):** Represents the potential truth (1 - evidence for the negation).
- **$U - L$ (Knowledge Gap):** Directly measures the model's **uncertainty** or ignorance regarding a specific rule or predicate.

### **Łukasiewicz Logic Gates**
We utilize differentiable logic gates. For example, the Conjunction (AND) is implemented as:
$$T(a, b) = \max(0, a + b - 1)$$
This allows the gradients to flow through logical operations, enabling the "backbone" to learn features that specifically satisfy the logical constraints.

### === ***Definition of Transformer block and Backbone*** ===

In [ ]:
class FuzzyMultiHeadSelfAttention(nnx.Module):
    """
    Fuzzy Multi-Head Self-Attention.

    Instead of the classic softmax attention score, it produces interval
    attention weights [L, U] — lower and upper bounds of fuzzy truth.
    The output is a pair (x_L, x_U) representing the fuzzy attention certainty.
    """
    def __init__(self, embed_dim: int, num_heads: int, uncertainty_margin: float = 0.1, rngs: nnx.Rngs = None):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.uncertainty_margin = uncertainty_margin  # width of fuzzy interval [L, U]

        self.qkv = nnx.Linear(embed_dim, 3 * embed_dim, rngs=rngs)
        self.proj_L = nnx.Linear(embed_dim, embed_dim, rngs=rngs)  # lower bound projection
        self.proj_U = nnx.Linear(embed_dim, embed_dim, rngs=rngs)  # upper bound projection

    def __call__(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = jnp.transpose(qkv, (2, 0, 3, 1, 4))  # (3, B, heads, N, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # ── Scaled dot-product scores ────────────────────────────────────────
        scale = self.head_dim ** -0.5
        scores = q @ jnp.transpose(k, (0, 1, 3, 2)) * scale  # (B, heads, N, N)

        # ── Fuzzy attention: softmax → interval [L, U] ───────────────────────
        # Mean attention value (classical softmax)
        attn_mid = jax.nn.softmax(scores, axis=-1)

        # Lower bound: "the least we know for sure" — sharp sigmoid shifted down
        attn_L = jax.nn.sigmoid(scores - self.uncertainty_margin)
        attn_L = attn_L / (attn_L.sum(axis=-1, keepdims=True) + 1e-6)

        # Upper limit: "the most we allow" — sigmoid shifted up
        attn_U = jax.nn.sigmoid(scores + self.uncertainty_margin)
        attn_U = attn_U / (attn_U.sum(axis=-1, keepdims=True) + 1e-6)

        # We guarantee L ≤ mid ≤ U (numerical fuse)
        attn_L = jnp.minimum(attn_L, attn_mid)
        attn_U = jnp.maximum(attn_U, attn_mid)

        # ── Fuzzy weighted sum over value vectors ────────────────────────────
        def attend(attn_weights):
            out = attn_weights @ v                                      # (B, heads, N, head_dim)
            return jnp.transpose(out, (0, 2, 1, 3)).reshape(B, N, C)   # (B, N, C)

        x_L = attend(attn_L)
        x_U = attend(attn_U)

        # ── Separate linear projections for L and U ────────────────────────────
        x_L = self.proj_L(x_L)  # (B, N, C)
        x_U = self.proj_U(x_U)  # (B, N, C)

        # Final output: midpoint of the interval for residual + interval for JLNN
        x_mid = (x_L + x_U) / 2.0

        return x_mid, jnp.stack([x_L, x_U], axis=-1)  # (B,N,C), (B,N,C,2)

In [ ]:
class EncoderBlock(nnx.Module):
    def __init__(self, embed_dim: int, mlp_dim: int, num_heads: int,
                 uncertainty_margin: float = 0.1, dropout_rate: float = 0.1, rngs: nnx.Rngs = None):
        super().__init__()
        self.norm1 = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.attn = FuzzyMultiHeadSelfAttention(embed_dim, num_heads, uncertainty_margin, rngs)
        self.norm2 = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.mlp = nnx.Sequential(
            nnx.Linear(embed_dim, mlp_dim, rngs=rngs),
            nnx.gelu,
            nnx.Linear(mlp_dim, embed_dim, rngs=rngs)
        )
        self.dropout = nnx.Dropout(rate=dropout_rate, rngs=rngs)

    def __call__(self, x, deterministic=False):
        # Fuzzy attention returns (mean, interval)
        attn_mid, attn_interval = self.attn(self.norm1(x))

        # Residual at the mean value (stable gradient)
        x = x + self.dropout(attn_mid, deterministic=deterministic)
        x = x + self.dropout(self.mlp(self.norm2(x)), deterministic=deterministic)

        return x, attn_interval  # we propagate the interval further for JLNN

In [ ]:
class ViTBackbone(nnx.Module):
    def __init__(self, patch_size=16, embed_dim=384, depth=6,
                 num_heads=6, mlp_dim=1536, uncertainty_margin=0.1, rngs=None):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim

        self.patch_embed = nnx.Conv(
            in_features=3, out_features=embed_dim,
            kernel_size=(patch_size, patch_size),
            strides=(patch_size, patch_size), rngs=rngs
        )

        self.cls_token = nnx.Param(jnp.zeros((1, 1, embed_dim)))
        num_patches = (224 // patch_size) ** 2
        self.pos_embed = nnx.Param(
            jax.random.truncated_normal(jax.random.PRNGKey(0), -2, 2,
                                        (1, num_patches + 1, embed_dim)) * 0.02
        )

        # Blocks as a list — Sequential cannot propagate double output
        self.blocks = nnx.List([
            EncoderBlock(embed_dim, mlp_dim, num_heads, uncertainty_margin, rngs=rngs)
            for _ in range(depth)
        ])
        self.norm = nnx.LayerNorm(embed_dim, rngs=rngs)

    def __call__(self, x, deterministic=False):
        B = x.shape[0]

        # 1. Patch embedding and transformation on a sequence
        x = self.patch_embed(x)
        x = x.reshape(B, -1, self.embed_dim)

        # 2. Adding CLS token and positional embedding
        cls_tokens = jnp.repeat(self.cls_token, B, axis=0)
        x = jnp.concatenate([cls_tokens, x], axis=1)
        x = x + self.pos_embed[...]

        # 3. Passing blocks — we collect fuzzy intervals from each block
        all_intervals = []
        for block in self.blocks:
            x, interval = block(x, deterministic=deterministic)
            all_intervals.append(interval)  # každé má tvar: (B, N, C, 2)

        x = self.norm(x)

        # 4. Calculating the average interval (FIX FOR ONNX)
        # Instead of jnp.mean(stacked, axis=0) we use sum/constant.
        # This ensures that the ONNX exporter does not use the problematic ReduceMean node with axes.
        stacked = jnp.stack(all_intervals, axis=0)        # (depth, B, N, C, 2)

        depth = len(self.blocks)
        mean_interval = jnp.sum(stacked, axis=0) / depth  # Stable averaging

        # Extract interval only for CLS token (index 0 in sequence)
        cls_interval = mean_interval[:, 0, :, :]          # (B, C, 2)

        return x[:, 0], cls_interval # Returns the CLS embedding and its fuzzy indeterminacy

### === ***JLNN Classifier*** ===

In [ ]:
class FuzzyGrounding(nnx.Module):
    def __init__(self, emb_dim: int, n_predicates: int, rngs: nnx.Rngs):
        super().__init__()

        # 1. Input stabilization (prevents sigmoid saturation at large values ​​of ViT)
        self.bn = nnx.BatchNorm(emb_dim, rngs=rngs)
        # 2. Linear projection with a cautious start
        # bias_init=-1.2 ensures that the predicates start in the region where the sigmoid has the largest gradient
        self.projection = nnx.Linear(
            emb_dim,
            n_predicates,
            bias_init=nnx.initializers.constant(-1.2), # A slight departure from the extreme
            rngs=rngs
        )

        # 3. Temperature (tau)
        # A value of 1.4 subtly sharpens decision-making but leaves room for learning
        self.tau = 1.4

    def __call__(self, x, cls_interval=None):
        # 1. CLS token normalization
        x = self.bn(x)

        # 2. Calculating logits with the influence of temperature
        logits = self.projection(x) / self.tau

        if cls_interval is not None:
            # We project uncertainty from the Attention mechanism (backbone)
            # cls_interval is usually of the form (B, 2) or (B, SeqLen, 2)
            # If averaged over a sequence:
            backbone_uncertainty = jnp.mean(cls_interval, axis=1)  # (B, 2)
            margin_L = backbone_uncertainty[:, 0:1]  # (B, 1)
            margin_U = backbone_uncertainty[:, 1:2]  # (B, 1)
        else:
            # Fixed small margin for stable start without backbone data
            margin_L = 0.1
            margin_U = 0.1

        # 3. Final calculation of [L, U] intervals using sigmoid
        # L (Lower bound) = conservative estimate of the truth
        # U (Upper bound) = maximum possible truth
        L = jax.nn.sigmoid(logits - margin_L)
        U = jax.nn.sigmoid(logits + margin_U)
        return jnp.stack([L, U], axis=-1)  # (B, n_predicates, 2)

In [ ]:
class JLNN_LogicMonitor(nnx.Module):
    def __init__(self, rules: list, predicate_names: list, rngs: nnx.Rngs):
        super().__init__()
        self.rules = nnx.List([LNNFormula(rule, rngs=rngs) for rule in rules])
        self.predicate_names = predicate_names

    def __call__(self, predicates):
        L = jnp.minimum(predicates[..., 0], predicates[..., 1])
        U = jnp.maximum(predicates[..., 0], predicates[..., 1])
        predicates = jnp.stack([L, U], axis=-1)  # (B, n_predicates, 2)

        # The median value of the interval as a scalar input to LearnedPredicate
        # LearnedPredicate expects (..., in_features=1), not an interval
        pred_mid = jnp.mean(predicates, axis=-1)  # (B, n_predicates) — average of L and U

        input_dict = {
            name: pred_mid[:, i:i+1]   # (B, 1) — one scalar per predicate
            for i, name in enumerate(self.predicate_names)
        }

        results = [rule(input_dict) for rule in self.rules]
        # Each rule (LNNFormula) returns (B, 2) — stack → (B, num_rules, 2)
        return jnp.stack(results, axis=1)

    def explain(self, audit_data, batch_idx=0):
        audit_data = np.array(audit_data)
        if audit_data.ndim == 2:
            audit_data = audit_data[None]

        print(f"{'Rule':<20} {'L':<8} {'U':<8} {'Certainty'}")
        print("-" * 55)

        for i, rule in enumerate(self.rules):
            L_val = float(audit_data[batch_idx, i, 0])
            U_val = float(audit_data[batch_idx, i, 1])

            certainty = (
                "✅ certain"   if L_val > 0.7 else
                "⚠️ uncertain" if U_val > 0.4 else
                "❌ false"
            )
            rule_name = str(rule)
            rule_name = rule_name[:18] + ".." if len(rule_name) > 18 else rule_name
            print(f"{rule_name:<20} {L_val:.4f}   {U_val:.4f}   {certainty}")

In [ ]:
class ViT_JLNN_Classifier(nnx.Module):
    def __init__(self,
                 vit_backbone: nnx.Module,
                 num_classes: int = 10,
                 n_predicates: int = 47,
                 rules: list = None,
                 predicate_names: list = None,
                 class_names: list = None,
                 rngs: nnx.Rngs = None):
        super().__init__()
        self.backbone = vit_backbone
        self.grounding = FuzzyGrounding(vit_backbone.embed_dim, n_predicates, rngs)
        self.logic = JLNN_LogicMonitor(rules, predicate_names, rngs) if rules is not None else None
        self.classifier_head = nnx.Linear(n_predicates, num_classes, rngs=rngs)
        self.class_names = class_names if class_names is not None else [str(i) for i in range(num_classes)]

    def get_class_name(self, idx):
        return self.class_names[idx]

    def __call__(self, x, deterministic=False):
        # Backbone now returns (cls_embedding, cls_interval)
        cls, cls_interval = self.backbone(x, deterministic=deterministic)

        # Grounding gets the interval from the backbone instead of a fixed margin
        predicates = self.grounding(cls, cls_interval=cls_interval)  # (B, n_predicates, 2)

        if self.logic is not None:
            audit = self.logic(predicates)
            logits = self.classifier_head(jnp.mean(predicates, axis=-1))
            return audit, predicates, logits
        else:
            logits = self.classifier_head(jnp.mean(predicates, axis=-1))
            return predicates, logits

### === ***Linking to the model*** ===

#### Rules (define before model!)

In [ ]:
rule_strings = [
    "0.75 :: (body & head & eyes & mouth) -> is_animal",
    "0.75 :: (metallic & structure & boxy & windows) -> is_vehicle",
    "0.82 :: (is_animal & four_legs & small_ears & whiskers & long_tail) -> is_cat",
    "0.88 :: (is_animal & two_legs & wings & beak & feathers & ~whiskers) -> is_bird",
    "0.80 :: (is_animal & four_legs & long_legs & hooves & short_tail & ~mane & antlers) -> is_deer",
    "0.81 :: (is_animal & four_legs & whiskers & ~long_tail & longer_snout) -> is_dog",
    "0.78 :: (is_animal & four_legs & colorful & shiny & bulging_eyes) -> is_frog",
    "0.82 :: (is_animal & four_legs & long_legs & hooves & mane & longer_neck) -> is_horse",
    "0.82 :: (is_vehicle & wheels & smaller) -> is_automobile",
    "0.85 :: (is_vehicle & wings & propeller) -> is_airplane",
    "0.87 :: (is_vehicle & wings & jet_engine) -> is_airplane",
    "0.88 :: (is_vehicle & water & ~wings & sails) -> is_ship",
    "0.86 :: (is_vehicle & water & ~wings & boat_engine) -> is_ship",
    "0.84 :: (is_vehicle & water & ~wings & chimney) -> is_ship",
    "0.83 :: (is_vehicle & wheels & ~smaller & longer) -> is_truck"
]

In [ ]:
print(f"✅ {len(rule_strings)} rules loaded")

#### Parameter definitions

In [ ]:
predicate_names = [
    "antlers", "beak", "boat_engine", "body", "boxy", "bulging_eyes", "chimney",
    "colorful", "eyes", "feathers", "four_legs", "head", "hooves", "is_airplane",
    "is_animal", "is_automobile", "is_bird", "is_cat", "is_deer", "is_dog",
    "is_frog", "is_horse", "is_ship", "is_truck", "is_vehicle", "jet_engine",
    "longer", "longer_neck", "longer_snout", "long_legs", "long_tail", "mane",
    "metallic", "mouth", "propeller", "sails", "shiny", "short_tail", "smaller",
    "small_ears", "structure", "two_legs", "water", "wheels", "whiskers",
    "windows", "wings"
]

In [ ]:
n_predicates = len(predicate_names)
print(f"Number of predicates: {n_predicates}")

In [ ]:
rngs = nnx.Rngs(42)

num_classes = 10
embed_dim = 384           # smaller = faster, less memory
depth = 6
num_heads = 6
mlp_dim = embed_dim * 4   # 1536

uncertainty_margin = 0.1            # width of fuzzy interval [L, U] in attention

#### 1. Creating a backbone

In [ ]:
vit_backbone = ViTBackbone(
    patch_size=16,
    embed_dim=embed_dim,
    depth=depth,
    num_heads=num_heads,
    mlp_dim=mlp_dim,
    uncertainty_margin=uncertainty_margin,
    rngs=rngs
)

#### 2. Creating the entire model

In [ ]:
model = ViT_JLNN_Classifier(
    vit_backbone=vit_backbone,
    num_classes=10,
    n_predicates=len(predicate_names),
    rules=rule_strings,
    predicate_names=predicate_names,
    class_names=class_names,
    rngs=rngs
)

In [ ]:
print("✅ ViT + JLNN model successfully created!")
print(f"   Embedding dim: {vit_backbone.embed_dim}")
print(f"   Fuzzy predicates: {model.grounding.n_predicates if hasattr(model.grounding, 'n_predicates') else 'N/A'}")

--------------------------------------------------
## ***Monitoring visualization (from Xarray)***
--------------------------------------------------

## **Training and Monitoring Strategy**

To ensure transparency during the learning process, we implement a multi-layered monitoring system:

1. **Xarray Auditing:** During training, we capture the internal state of logical rules. These are saved into `.nc` (NetCDF) files, allowing us to visualize how the model's "reasoning" evolves as it sees more data.
2. **Hardware Profiling:** We track **Images Per Second (Throughput)** to optimize the utilization of JAX on TPU/GPU accelerators.
3. **TensorBoard Integration:** Real-time visualization of loss curves, accuracy, and logic stability metrics.

In [ ]:
def plot_logic_monitoring(nc_path, smooth_window=5):
    """
    Visualization of rule evolution with smoothing and unique colors.
    smooth_window: size of the smoothing window (the higher the smoother the curves)
    """
    ds = xr.open_dataset(nc_path)
    # Average over batch, lower bound L (lu index 0)
    # data shape: (time, rule)
    raw_data = ds.truth_values.mean(dim="batch")[:, :, 0].values
    steps = np.arange(raw_data.shape[0])
    rule_names = ds.rule.values
    num_rules = len(rule_names)

    # 1. Color settings - we will generate unique colors from the viridis or tab20 palette
    # If you have less than 20 rules, tab20 is very contrasting.
    # For more rules, 'jet' or 'nipy_spectral' is better.
    cmap = plt.get_cmap('tab20' if num_rules <= 20 else 'nipy_spectral')
    colors = [cmap(i) for i in np.linspace(0, 1, num_rules)]

    plt.figure(figsize=(13, 7))

    for i, rule_name in enumerate(rule_names):
        y = raw_data[:, i]

        # 2. Smoothing using moving average (pandas)
        if smooth_window > 1:
            y_smooth = pd.Series(y).rolling(window=smooth_window, min_periods=1, center=True).mean()
        else:
            y_smooth = y

        plt.plot(steps, y_smooth, label=rule_name, color=colors[i], linewidth=2, alpha=0.85)
        # Optional: render the original data very faintly in the background
        # plt.plot(steps, y, color=colors[i], alpha=0.1)

    plt.title(f"Evolution of Rule Truth Values (L) - Smoothed (window={smooth_window})", fontsize=14)
    plt.xlabel("Step (Training Progress)", fontsize=12)
    plt.ylabel("Fuzzy Truth Value (Lower Bound L)", fontsize=12)

    # Legend on the right with better formatting
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=9, ncol=1 if num_rules <= 20 else 2)

    #plt.ylim(-0.05, 1.05) # Fuzzy value range
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

-----------------------------------
## ***TRAINING SETUP***
-----------------------------------

### === ***Learning rate scheduler s warmup*** ===

In [ ]:
DEMO_MODE = False

In [ ]:
num_epochs = 2 if DEMO_MODE else 20
steps_per_epoch = 200 if DEMO_MODE else 1000

In [ ]:
total_steps  = num_epochs * steps_per_epoch
warmup_steps = 100 if DEMO_MODE else max(1000, total_steps // 10)

In [ ]:
print(f"total_steps:  {total_steps}")
print(f"warmup_steps: {warmup_steps}")

In [ ]:
lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=1e-4,
    warmup_steps=warmup_steps,
    decay_steps=total_steps,
    end_value=1e-6
)

### === ***Optimizer*** ===

In [ ]:
optimizer = nnx.Optimizer(
    model,
    optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(lr_schedule, weight_decay=0.01)
    ),
    wrt=nnx.Param
)

### === ***JIT-ed train step*** ===

In [ ]:
@nnx.jit
def train_step(model, optimizer, batch):
    x = batch['image']  # (B, 224, 224, 3)
    y = batch['label']  # (B,)

    def loss_fn(model):
        # deterministic=False → Dropout active during training
        outputs = model(x, deterministic=False)

        if isinstance(outputs, tuple):
            if len(outputs) == 3:
                audit, predicates, logits = outputs
            else:
                predicates, logits = outputs
        else:
            logits = outputs

        one_hot = jax.nn.one_hot(y, num_classes)
        loss = optax.softmax_cross_entropy(logits=logits, labels=one_hot).mean()

        return loss, logits

    (loss, logits), grads = nnx.value_and_grad(loss_fn, has_aux=True)(model)
    optimizer.update(model, grads)

    return loss, logits

### === ***Simple eval step (for validation)*** ===

In [ ]:
@nnx.jit
def eval_step(model, batch):
    x = batch['image']
    y = batch['label']

    # deterministic=True → Dropout disabled during validation/inference
    outputs = model(x, deterministic=True)

    if isinstance(outputs, tuple):
        if len(outputs) == 3:
            _, _, logits = outputs
        else:
            _, logits = outputs
    else:
        logits = outputs

    one_hot = jax.nn.one_hot(y, num_classes)
    loss = optax.softmax_cross_entropy(logits=logits, labels=one_hot).mean()
    acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)

    return loss, acc

### === ***Initializing Checkpointing (Before Training)*** ===

In [ ]:
checkpoint_dir = os.path.abspath('./checkpoints')

Checkpoint manager for PyTree (nnx.Module and optimizer are PyTrees)

In [ ]:
options = ocp.CheckpointManagerOptions(max_to_keep=3, create=True)

In [ ]:
checkpoint_manager = ocp.CheckpointManager(
    ocp.test_utils.erase_and_create_empty(checkpoint_dir), # Smaže staré při novém startu
    options=options
)

Load the last checkpoint, if any

In [ ]:
latest = checkpoint_manager.latest_step()
start_epoch = 0
if latest is not None:
    ckpt = checkpoint_manager.restore(
        latest,
        args=ocp.args.StandardRestore({'model': model, 'optimizer': optimizer})
    )
    start_epoch = ckpt.get('step', 0)
    if 'model' in ckpt:
        model.update(ckpt['model'])
    print(f"✅ Restored from step {latest}, resuming epoch {start_epoch}")
else:
    print("🚀 Starting training from scratch.")

### === ***Training loop*** ===

Buffer for monitoring logic

In [ ]:
writer = SummaryWriter(log_dir="./logs")

In [ ]:
%load_ext tensorboard

In [ ]:
print(f"🚀 Starting training...")

MAX_MONITOR_SAMPLES = 50

for epoch in range(start_epoch, num_epochs):
    epoch_loss = 0.0
    start_time = time.time()
    monitor_history = []
    last_batch = None  # we will keep the last batch for explain

    # ── TRAIN LOOP ──────────────────────────────────────────────────────────
    model.train()
    for step, batch in enumerate(tqdm(train_ds, total=steps_per_epoch, desc=f"Epoch {epoch+1} [train]")):
        if step >= steps_per_epoch:
            break

        # --- HARDWARE PROFILING (XProf) START ---
        # We capture steps 50-60 in the first epoch for detailed performance analysis
        if epoch == 0 and step == 50:
            jax.profiler.start_trace("./logs")
            print("\n🔍 XProf: Starting hardware trace (steps 50-60)...")
        # ----------------------------------------

        step_start = time.time()

        loss, logits = train_step(model, optimizer, batch)
        epoch_loss += float(loss)

        # --- HARDWARE PROFILING (XProf) STOP ---
        if epoch == 0 and step == 60:
            jax.profiler.stop_trace()
            print("\n✅ XProf: Trace finished. Logging to TensorBoard Profile tab.")
        # ---------------------------------------

        global_step = epoch * steps_per_epoch + step
        if step % 10 == 0:
            throughput = batch_size / (time.time() - step_start)
            writer.add_scalar("Loss/train", float(loss), global_step)
            writer.add_scalar("Perf/Throughput_imgs_sec", throughput, global_step)

        if step % 20 == 0 and len(monitor_history) < MAX_MONITOR_SAMPLES:
            outputs = model(batch['image'], deterministic=True)
            audit, predicates, _ = outputs
            monitor_history.append({'audit': np.array(audit)})

        last_batch = batch  # we are constantly updating

    avg_train_loss = epoch_loss / steps_per_epoch

    # ── VALIDATION LOOP ─────────────────────────────────────────────────────
    model.eval()
    val_losses, val_accs = [], []

    for val_batch in tqdm(val_ds, desc=f"Epoch {epoch+1} [val]  "):
        v_loss, v_acc = eval_step(model, val_batch)
        val_losses.append(float(v_loss))
        val_accs.append(float(v_acc))

    avg_val_loss = np.mean(val_losses)
    avg_val_acc  = np.mean(val_accs)

    writer.add_scalar("Loss/val", avg_val_loss, epoch)
    writer.add_scalar("Acc/val",  avg_val_acc,  epoch)
    writer.add_scalar("Loss/train_avg", avg_train_loss, epoch)

    elapsed = time.time() - start_time
    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"train_loss={avg_train_loss:.4f} | "
        f"val_loss={avg_val_loss:.4f} | "
        f"val_acc={avg_val_acc:.4f} | "
        f"time={elapsed:.1f}s"
    )

    # ── FUZZY LOGIC EXPLAIN (1× per epoch, outside JIT) ────────────────────────
    if last_batch is not None:
        # Get fresh outputs from the model (outside JIT, so audit is a plain array)
        outputs = model(last_batch['image'], deterministic=True)
        audit_jax, _, _ = outputs

        print(f"\n--- Fuzzy audit epoch {epoch+1} (sample 0) ---")

        # Convert JAX array to NumPy and pass directly to explain
        audit_np = np.array(audit_jax)
        print(f"DEBUG audit_np.shape: {audit_np.shape}, dtype: {audit_np.dtype}")
        model.logic.explain(audit_np, batch_idx=0)

    # ── END OF EPOCH ─────────────────────────────────────────────────────────

    checkpoint_manager.save(
        epoch + 1,
        args=ocp.args.StandardSave({'model': model, 'optimizer': optimizer})
    )

    checkpoint_manager.wait_until_finished()

    if monitor_history:
        all_audits = jnp.stack([m['audit'] for m in monitor_history])
        ds = xr.Dataset(
            {"truth_values": (("time", "batch", "rule", "lu"), all_audits)},
            coords={"rule": [f"rule_{i}" for i in range(len(rule_strings))]}
        )
        current_log = f"logic_epoch_{epoch+1}.nc"
        ds.to_netcdf(current_log)

        # --- VISUALIZATION OF THE CURRENT EPOCLE ---
        # Display the rule stability graph immediately after saving
        print(f"\n📊 Logic stability monitoring for epoch {epoch+1}:")
        try:
            plot_logic_monitoring(current_log)
        except Exception as e:
            print(f"⚠️ Failed to render monitoring: {e}")
        # -----------------------------------

    try:
        save_to_hf(epoch + 1)
    except Exception as e:
        print(f"⚠️ HF upload failed: {e}")

In [ ]:
%tensorboard --logdir ./logs

--------------------------------------------------
## ***Export model***
--------------------------------------------------

## **Model Export and Production Readiness**

The final step involves freezing the model state and exporting it for external inference:
- **StableHLO:** A portable format for high-performance execution in XLA-compatible runtimes.

### === ***Export to StableHLO*** ===

In [ ]:
def to_pure(x):
    if isinstance(x, (dict, nnx.State)):
        return {k: to_pure(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return type(x)(to_pure(v) for v in x)
    if isinstance(x, nnx.Variable):
        # Místo x.value použijte x[...] (pro pole) nebo x.get_value()
        return x[...]
    return x

In [ ]:
def prepare_model_for_export(model):
    """Prepare the model for export — disable dropout and RNG"""
    print("🔧 Preparing model for export (disabling stochastic layers)...")

    count = 0
    for path, node in nnx.iter_graph(model):
        if hasattr(node, 'deterministic'):
            node.deterministic = True
            count += 1
        if hasattr(node, 'rngs') and isinstance(getattr(node, 'rngs', None), dict):
            if 'dropout' in node.rngs:
                node.rngs['dropout'] = None

    print(f"   ✅ Set deterministic=True on {count} layers")
    print("   ✅ Model prepared for export")
    return model

In [ ]:
def export_to_stablehlo(model: nnx.Module, sample_input):
    print("   → Splitting and purifying NNX model...")
    graphdef, state = nnx.split(model)

    # 1. Convert the state to pure dictation (remove Param/Variable nodes)
    pure_state = to_pure(state)
    flat_params, pure_treedef = jax.tree.flatten(pure_state)

    @jax.jit
    def forward_fn(flat_p, x):
        # 2. Reconstruction from pure JAX types
        params_dict = jax.tree.unflatten(pure_treedef, flat_p)
        # 3. Merge back - nnx.merge natively handles field dictation if GraphDef fits
        m = nnx.merge(graphdef, params_dict)
        # 4. Deterministic pass + output purification (important!)
        out = m(x, deterministic=True)
        return to_pure(out)

    # Abstract values ​​for flat parameters
    flat_avals = [jax.ShapeDtypeStruct(p.shape, p.dtype) for p in flat_params]
    input_aval = jax.ShapeDtypeStruct(sample_input.shape, sample_input.dtype)

    print("   → Lowering to StableHLO (purified tree)...")
    # We call the export directly from the jax module
    exported = jax.export.export(forward_fn)(flat_avals, input_aval)

    print("✓ StableHLO export successful")
    return exported, {"treedef": pure_treedef, "graphdef": graphdef}

In [ ]:
def save_stablehlo_artifact(exported_tuple, path: str):
    if isinstance(exported_tuple, tuple):
        exported, _ = exported_tuple
    else:
        exported = exported_tuple

    serialized_bytes = exported.serialize()

    with open(path, "wb") as f:
        f.write(serialized_bytes)

    size_kb = len(serialized_bytes) / 1024
    print(f"✓ StableHLO artifact saved to: {path} ({size_kb:.2f} KB)")

#### ***INFERENCE WRAPPER***

In [ ]:
def check_stablehlo_inference(artifact_path, model, sample_input):
    """
    Loads StableHLO and performs inference with the correct arguments (params, input).
    """
    # 1. Retrieving the artifact
    with open(artifact_path, "rb") as f:
        serialized_bytes = f.read()
    exported_model = jax.export.deserialize(serialized_bytes)

    # 2. Getting parameters from the current model (same way as when exporting)
    # We need to use to_pure to get pure arrays (JAX arrays)
    def to_pure(x):
        if isinstance(x, (dict, nnx.State)):
            return {k: to_pure(v) for k, v in x.items()}
        if isinstance(x, (list, tuple)):
            return type(x)(to_pure(v) for v in x)
        if isinstance(x, nnx.Variable):
            return x[...]
        return x

    _, state = nnx.split(model)
    pure_state = to_pure(state)
    # flat_params must be in the same order as when exporting (jax.tree.flatten)
    flat_params, _ = jax.tree.flatten(pure_state)

    # 3. Preparing the entrance
    input_data = jnp.array(sample_input, dtype=jnp.float32)

    # 4. CALL WITH TWO ARGUMENTS (parameters, input)
    # forward_fn(flat_params, x)
    results = exported_model.call(flat_params, input_data)

    return results

#### ***STARTING EXPORT***

In [ ]:
def full_export_pipeline(model: nnx.Module, test_ds, base_name="vit_jlnn_final"):
    """Complete export: StableHLO"""

    sample_batch = next(iter(test_ds))
    sample_input = sample_batch['image'][:1]   # ← tvar (1, 224, 224, 3)

    print("🔧 Preparing model...")
    model_exp = prepare_model_for_export(model)

    # StableHLO
    print("📦 1. Exporting to StableHLO...")
    exported_tuple = export_to_stablehlo(model_exp, sample_input)
    save_stablehlo_artifact(exported_tuple, f"{base_name}.stablehlo")

    print("\n🎉 Export pipeline finished!")

In [ ]:
full_export_pipeline(model, test_ds, base_name="vit_jlnn_final")

#### ***Inference example***

In [ ]:
sample_batch = next(iter(test_ds))
input_img = sample_batch['image'][:1]

output = check_stablehlo_inference("vit_jlnn_final.stablehlo", model, sample_batch['image'][:1])
print(f"Inference result (Logits):\n {output[2]}\n")
print(f"JLNN audit:\n {output[0]}")

---------------------------------------
## **Saving all training files**
---------------------------------------

In [ ]:
api = HfApi()
repo_id = "KRadim/vit-jlnn-cifar10"

print(f"🚀 Starting a comprehensive upload to {repo_id}...")

# 1. Upload the main StableHLO model
if os.path.exists("vit_jlnn_final.stablehlo"):
    api.upload_file(
        path_or_fileobj="vit_jlnn_final.stablehlo",
        path_in_repo="vit_jlnn_final.stablehlo",
        repo_id=repo_id,
        commit_message="Update StableHLO model"
    )

# 2. Upload folders (Checkpoints and Logs) - uploads the entire content recursively
for folder in ["checkpoints", "logs"]:
    if os.path.exists(folder):
        print(f"  → Nahrávám složku {folder}...")
        api.upload_folder(
            folder_path=folder,
            path_in_repo=folder,
            repo_id=repo_id,
            commit_message=f"Update {folder} data"
        )

# 3. Upload all xarray (.nc) files from the root directory
nc_files = [f for f in os.listdir('.') if f.endswith('.nc')]
for nc_file in nc_files:
    api.upload_file(
        path_or_fileobj=nc_file,
        path_in_repo=nc_file,
        repo_id=repo_id,
        commit_message=f"Upload xarray log: {nc_file}"
    )

print("\n🎉 All components (StableHLO, Checkpoints, Logs and .nc files) have been successfully saved to Hugging Face.")

-----------------------------------------------
## **PACKAGING AND DOWNLOADING RESULTS**
-----------------------------------------------

In [ ]:
zip_name = "jlnn_training_export"
bundle_dir = "export_bundle"

In [ ]:
print(f"📦 I am preparing a download package...")
if os.path.exists(bundle_dir): shutil.rmtree(bundle_dir)
os.makedirs(bundle_dir, exist_ok=True)

# List of folders and files we want to ZIP
items_to_include = ['checkpoints', 'logs', 'vit_jlnn_final.onnx', 'vit_jlnn_final.stablehlo']
# Add all xarray (.nc) logs
nc_files = [f for f in os.listdir('.') if f.endswith('.nc')]
items_to_include.extend(nc_files)

for item in items_to_include:
    if os.path.exists(item):
        if os.path.isdir(item):
            shutil.copytree(item, os.path.join(bundle_dir, item), dirs_exist_ok=True)
        else:
            shutil.copy2(item, bundle_dir)

# Create a ZIP archive
shutil.make_archive(zip_name, 'zip', bundle_dir)
print(f"✅ Archive created: {zip_name}.zip")

# Automatically start downloads in the browser
files.download(f"{zip_name}.zip")